# 04 - Question-Answering Interface (Gradio)

**Goal:** run the Text-to-SQL question-answering UI with the fine-tuned model. Features:
light/dark theme, SQL with syntax highlighting and a copy button, chat history, guardrail badge
(safe/rejected + reason), read-only execution on sample databases (Online Store, Campus, Hospital) with results shown as a table,
natural-language answers written by the model and checked against the data, automatic retry when a query fails,
questions in any language (translated automatically), a custom-schema mode (syntax validation only),
and uploading your own CSV / SQLite data.

**Google Colab:** *Runtime -> Change runtime type -> T4 GPU*. The first cell looks for the project zip (Google Drive, or an upload dialog). Run the cells from top to bottom (*Runtime -> Run all*). Save the notebook together with its outputs: *File -> Download -> Download .ipynb*.
**Kaggle:** Accelerator = T4 GPU (not P100), Internet = On, project added as a Dataset (*Add Input*).

The internet must be on (public Gradio link). Empty `MODEL_ID` = the adapter is located automatically on HF (repo `<token-username>/qwen2.5-coder-3b-text2sql-qlora`).
Once the UI appears, take 2-3 **real** screenshots (see `screenshots/README.md`).

In [1]:
import glob
import os
import sys
from pathlib import Path

try:                                    # detect Google Colab
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = True   # Colab: keep data/results on Google Drive so they survive a disconnected session


def find_project_root():
    """Find the project folder (the one containing src/sql_utils.py): working dir, Kaggle Dataset, or a Colab upload."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    candidates += [Path(p).parent.parent for p in glob.glob("/kaggle/input/**/src/sql_utils.py", recursive=True)]
    candidates += [Path(p).parent.parent for p in glob.glob("/content/*/src/sql_utils.py") + glob.glob("/content/*/*/src/sql_utils.py")]
    for c in candidates:
        if (c / "src" / "sql_utils.py").exists():
            return c
    return None


ROOT = find_project_root()
if ROOT is None and IN_COLAB:
    import zipfile
    ZIP_NAME = "Text2SQL_Dwi-Cahyo-Mariyanto.zip"
    # Easiest: put the project zip ONCE in Google Drive (My Drive); this cell finds it by itself, no upload dialog.
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            for z in (Path("/content/drive/MyDrive") / ZIP_NAME, Path("/content/drive/MyDrive/Text2SQL") / ZIP_NAME):
                if z.exists():
                    zipfile.ZipFile(z).extractall("/content")
                    print("Project loaded from Google Drive:", z)
                    break
        except Exception as e:                      # fine: fall back to manual upload
            print("Google Drive not used:", type(e).__name__)
        ROOT = find_project_root()
    if ROOT is None:                                # fallback: upload the zip from your computer
        from google.colab import files
        print("Select the file", ZIP_NAME, "from your computer ...")
        for name in files.upload():
            if name.lower().endswith(".zip"):
                zipfile.ZipFile(name).extractall("/content")
        ROOT = find_project_root()
if ROOT is None:
    raise FileNotFoundError(
        "Project folder not found. Colab: run this cell again and upload the project zip. "
        "Kaggle: upload the zip as a Dataset, then 'Add Input'."
    )

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # use a single GPU (safe on T4 x2)
if IN_COLAB and USE_DRIVE:
    try:
        if not os.path.ismount("/content/drive"):
            from google.colab import drive
            drive.mount("/content/drive")
        os.environ["T2S_WORK_DIR"] = "/content/drive/MyDrive/Text2SQL_Dwi-Cahyo-Mariyanto"
    except Exception as e:                          # fine: results are kept locally in the Colab session
        print("Google Drive not mounted, results are kept locally in this session:", type(e).__name__)
for sub in ("src", "Fine-tuning_submission", "Evaluation_submission", "Interface_submission"):
    sys.path.insert(0, str(ROOT / sub))
print("Project root:", ROOT)
if IN_COLAB:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT ACTIVE -> Runtime > Change runtime type > T4 GPU")

Mounted at /content/drive
Select the file Text2SQL_Dwi-Cahyo-Mariyanto.zip from your computer ...


Saving Text2SQL_Dwi-Cahyo-Mariyanto-final.zip to Text2SQL_Dwi-Cahyo-Mariyanto-final.zip
Project root: /content/Text2SQL_Dwi-Cahyo-Mariyanto
GPU: Tesla T4


In [2]:
%pip install -q gradio unsloth  # [Kaggle/Colab] skip if already installed

## 1. Configuration

In [16]:
MODEL_ID = ""            # empty = automatic; or "HF-username/repo" / adapter folder path
USE_MOCK = False         # True = rule-based mock model, ONLY for trying the UI layout without a GPU
KEEP_ALIVE_MINUTES = 10  # how long the UI stays alive; set 0 for 'Save & Run All' so the notebook finishes by itself

**Hugging Face token (manual):** run the token cell below. If `HF_TOKEN` is not found in Secrets, a hidden input box appears: paste your token (*Write* permission for notebook 02, *Read* is enough for 03/04) and press Enter. Never write the token into a cell; keep `HF_TOKEN_MANUAL` empty before sharing the notebook.

In [17]:
from eval_utils import ensure_hf_token, resolve_adapter

HF_TOKEN_MANUAL = ""   # optional; leave empty to be asked through a hidden input box
if not USE_MOCK:
    HF_TOKEN = ensure_hf_token(HF_TOKEN_MANUAL)
    MODEL_ID = resolve_adapter({"adapter": MODEL_ID, "hub_repo_name": "qwen2.5-coder-3b-text2sql-qlora"}, HF_TOKEN)
    print("Model in use:", MODEL_ID)

Logged in to Hugging Face as: dwicahyom16
Model in use: dwicahyom16/qwen2.5-coder-3b-text2sql-qlora


## 2. App logic test (CPU, mock model)
Checks that the *generate -> guardrail -> read-only execution* flow works before the real model is loaded.
The mock model is **not** a real model output; it only proves the flow and the guardrail.

In [18]:
from app import CUSTOM, MockGenerator, answer_question

mock = MockGenerator()
for q in ["How many customers are there?", "Please delete the whole table"]:
    t = answer_question(mock, "Online Store", "", q)
    print(f"Q: {q}\n   SQL: {t.sql}\n   Guardrail: {t.guard.allowed} - {t.guard.reason}\n   Result: {t.rows[:3]} | {t.note}\n")

Q: How many customers are there?
   SQL: SELECT COUNT(*) FROM customers
   Guardrail: True - Safe: read-only query (SELECT).
   Result: [(20,)] | 1 row(s) (max 100).

Q: Please delete the whole table
   SQL: DROP TABLE customers
   Guardrail: False - Only SELECT queries are allowed.
   Result: [] | The query was not executed.



## 3. Load the model & launch the UI **[GPU]**

In [24]:
import time
from app import HFGenerator, build_ui, launch_ui

assert USE_MOCK or MODEL_ID, "Set MODEL_ID (or USE_MOCK = True to try the layout only)."
generator = MockGenerator() if USE_MOCK else HFGenerator(MODEL_ID)
demo = build_ui(generator)
launch_ui(demo, share=True, prevent_thread_lock=True)   # share=True -> temporary public link

==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1d194364c6ead73de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
# Keep the UI alive so it can be tried and screenshotted. Set KEEP_ALIVE_MINUTES = 0 for Save & Run All.
if KEEP_ALIVE_MINUTES:
    print(f"UI alive for {KEEP_ALIVE_MINUTES} minutes...")
    time.sleep(KEEP_ALIVE_MINUTES * 60)
demo.close()

## 4. What to try for the screenshots
1. Pick the *Hospital* (or *Online Store*) database, click an example question, press **Send** (shows SQL + result table + answer).
2. Ask something dangerous (e.g. *"Delete all customers"*) and watch the guardrail badge.
3. Pick *My data (upload CSV / SQLite)* and upload `sales.csv`, then ask a question about it.